# A.4 — Loss DPO từ bốn log-xác suất chuỗi
Chạy trên CPU với repo và PyTorch như A.1. Đây là **ví dụ minh họa xác định**, không phải số liệu C.4 của giảng viên, cặp khảo sát, hay huấn luyện ở D. Nguồn: [DPO, §4, Eq. (4)–(7)](https://arxiv.org/abs/2305.18290), [chứng minh B.5](../docs/theory/part-b.pdf).

## Từ xác suất ưu tiên đến loss
Với $\beta>0$, trên hỗ trợ của reference và khi hằng số chuẩn hóa hữu hạn, nghiệm của bài toán tối đa hóa phần thưởng có phạt KL thỏa $\pi^*(y|x)\propto\pi_{ref}(y|x)\exp(r(x,y)/\beta)$. Do đó có thể tham số hóa phần thưởng dưới dạng $r_\theta(x,y)=\beta[\log\pi_\theta(y|x)-\log\pi_{ref}(y|x)]+c(x)$. Trong so sánh hai đáp án **cùng prompt**, $c(x)$ triệt tiêu. Bradley–Terry cho $P(w\succ l|x)=\sigma(z)$ với $z=\beta[(p_w-p_l)-(r_w-r_l)]$, trong đó $p,r$ lần lượt là **log-xác suất** của policy và reference. Tối đa hóa likelihood của nhãn ưu tiên tương đương cực tiểu $L=-\frac1B\sum_i\log\sigma(z_i)$.

`src/dpo.py` dùng `logsigmoid` để không tính `log(sigmoid(z))` trực tiếp ở biên âm lớn. Mỗi số đầu vào là $\sum_{t\in\mathrm{đáp\ án}}\log\pi(y_t|x,y_{<t})$: căn đúng token kế tiếp, bỏ prompt và padding, **không chia cho độ dài**. Chỉ bước cuối lấy trung bình theo **cặp**. Khi nối D, lấy logprob từ cùng tokenizer/mask và reference SFT đóng băng (tính dưới `torch.no_grad()`); hàm loss cũng tách gradient reference.

In [ ]:
from pathlib import Path
import math
import platform
import subprocess
import sys
import torch

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/dpo.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Hãy chuyển thư mục làm việc đến repo có src/dpo.py và tests/.')
sys.path.insert(0, str(root))
from src.dpo import dpo_loss

print(f'Python {platform.python_version()}, torch {torch.__version__}, CPU; dữ liệu cố định')

## Tính tay trước khi chạy
Chọn $(p_w,p_l)=(-2,-3)$, $(r_w,r_l)=(-4,-5)$ và $\beta=0.5$. Hai độ chênh bằng nhau, $z=0$, nên $P=1/2$, $L=\ln 2$. Với $B=1$, $\partial L/\partial p_w=-\beta\sigma(-z)$ và $\partial L/\partial p_l=+\beta\sigma(-z)$: ở đây là $-0.25$ và $+0.25$. Khi hạ gradient, policy tăng ưu tiên cho đáp án được chọn. Nếu tăng riêng $p_w$ lên $-1$, loss phải giảm. Với $B>1$ các đạo hàm trên còn chia cho $B$.

In [ ]:
pw = torch.tensor(-2., dtype=torch.float64, requires_grad=True)
pl = torch.tensor(-3., dtype=torch.float64, requires_grad=True)
rw, rl = torch.tensor([-4., -5.], dtype=torch.float64).unbind()
loss = dpo_loss(pw, pl, rw, rl, beta=0.5)
gw, gl = torch.autograd.grad(loss, (pw, pl))
better = dpo_loss(pw + 1, pl, rw, rl, beta=0.5)
assert math.isclose(loss.item(), math.log(2)) and better.item() < loss.item()
assert math.isclose(gw.item(), -0.25) and math.isclose(gl.item(), 0.25)
print(f'Loss ban đầu: {loss.item():.8f}; gradient: ({gw.item():+.4f}, {gl.item():+.4f}); loss sau: {better.item():.8f}')

## Đối chiếu bằng PyTorch
Bộ kiểm dùng cross-entropy hai lớp `[chosen, rejected]` với nhãn 0 làm phép tính độc lập. So loss trung bình và cả hai gradient policy trên minibatch có chênh lệch âm, dương và 0; kiểm reference không nhận gradient và độ ổn định ở biên lớn. Đối chiếu trên minibatch **thật** ở D vẫn cần làm khi có dữ liệu hợp lệ.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_dpo.py', '-v'], cwd=root, check=True);